# TLT debug diagnostics

**Before:** `realize()` failed with a generic root-level message no path, no error code.

**After:** `explain(impl)` walks the tree and reports *where* the problem is.

Run **Kernel → Restart & Run All**.

In [1]:
import sys
sys.path.insert(0, "../src")

from pyspect import *
from pyspect.impls.str import StrImpl

impl = StrImpl(['x'])

## 1) Deep tree invalid approximation

**Before:**
```
Exception: Invalid approximation. TLT operational semantics of formula does not hold.
```

**After:**

In [3]:
TLT.select(ContLTL)

progress = UNTIL(AND('lane', NOT('obstacle')), 'goal')
safe_mission = AND(progress, 'battery_ok')  # intentional bug
phi = AND('mission_enabled', OR(safe_mission, 'emergency_stop'))

tree = TLT(phi, where={
    'lane': Set('LANE'), 'obstacle': Set('OBSTACLE'), 'goal': Set('GOAL'),
    'battery_ok': Set('BATTERY_OK'), 'emergency_stop': Set('EMERGENCY_STOP'),
    'mission_enabled': Set('MISSION_ON'),
})

print(tree.explain(impl))

TLT diagnostic report:
  [INVALID_APPROX_COMPOSITION] at root.right.left
  'AND' incompatible approximations (UNDER, EXACT).


## 2) Missing proposition binding

**Before:** `Exception: Missing proposition \`goal\``

**After:**

In [4]:
TLT.select(ContLTL)
tree = TLT(UNTIL('safe', 'goal'), where={'safe': Set('SAFE')})
print(tree.explain(impl))

TLT diagnostic report:
  [MISSING_PROPOSITION_BINDING] at root.right
  Proposition 'goal' is not bound.


## 3) Implementation missing an operation

**Before:** `Exception: Missing from implementation: pre`

**After:**

In [5]:
TLT.select(DiscLTL)
tree = TLT(NEXT('a'), where={'a': Set('A')})
print(tree.explain(impl))

TLT diagnostic report:
  [MISSING_IMPL_OPERATION] at root
  StrImpl missing 'pre' (needs 'F-NEXT').


## 4) Good path

In [6]:
TLT.select(ContLTL)

phi = UNTIL(AND('lane', NOT('obstacle')), 'goal')
tree = TLT(phi, where={'lane': Set('LANE'), 'obstacle': Set('OBSTACLE'), 'goal': Set('GOAL')})

print(tree.explain(impl))
print(tree.realize(impl))

No issues detected.
Reach(GOAL, (LANE ∩ OBSTACLEꟲ))


## 5)  deep-tree example

Bug at `root.right.right`: `AND(UNTIL(...), 'c')` mixes `UNDER` and `EXACT`.

```
root (AND)
├── root.left                    mission_on
└── root.right (OR)
    ├── root.right.left          backup
    └── root.right.right (AND)   ← error here
        ├── UNTIL(a, b)          UNDER
        └── c                    EXACT
```

In [2]:
TLT.select(ContLTL)

bad = AND(UNTIL('a', 'b'), 'c')
middle = OR('backup', bad)
phi = AND('mission_on', middle)

# phi = OR('backup', UNTIL(AND(AND('a', 'c'), 'mission_on'), 'b'))  # fix

tree = TLT(phi, where={
    'mission_on': Set('ON'),
    'backup': Set('BACKUP'),
    'a': Set('A'), 'b': Set('B'), 'c': Set('C'),
})

print(tree.explain(impl))

TLT diagnostic report:
  [INVALID_APPROX_COMPOSITION] at root.right.right
  'AND' incompatible approximations (UNDER, EXACT).


1. **Build**  `UNTIL` → `UNDER`, `c` → `EXACT`, their `AND` → `INVALID` at `root.right.right`
2. **Walk**  `_walk()` visits every node
3. **Deepest**  reports `root.right.right`, not `root` (the cascade above is just a consequence)
4. **Report**  `explain()` prints path + cause

**Full fix** (not just `bad`): fold props inside `UNTIL`, drop the outer `AND` with a `UNTIL` child